## 📘 Entrenamiento PPO con TF-Agents en el entorno SuperMarioBros-v3

Este notebook entrena un agente PPO en el entorno `SuperMarioBros-v3` usando `TF-Agents`, y cumple con los requisitos de:
- Comparación con política aleatoria
- Visualización en video del agente entrenado
- Gráfica de recompensa durante el entrenamiento

### ✅ Paso 1: Instalación de dependencias

In [ ]:
!pip install "tensorflow==2.15.0" tf-agents==0.19.0 \
    tensorflow-probability==0.22.0 gym==0.26.2 gym-notices==0.0.8 \
    gym_super_mario_bros==7.4.0 nes_py==8.2.1 \
    pyvirtualdisplay==3.0 pyglet==1.5.27 \
    imageio==2.34.0 imageio-ffmpeg==0.4.9

import os
os.kill(os.getpid(), 9)  # Reiniciar entorno para aplicar cambios

### ✅ Paso 2: Wrappers y creación del entorno

In [ ]:
import collections
import numpy as np
import gym
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from gym.wrappers import GrayScaleObservation, ResizeObservation

# Normalización y apilamiento
class NormalizeObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0,
            shape=env.observation_space.shape,
            dtype=np.float32
        )

    def observation(self, observation):
        return observation.astype(np.float32) / 255.0

class FixedFrameStack(gym.Wrapper):
    def __init__(self, env, num_frames):
        super().__init__(env)
        self.num_frames = num_frames
        self.frames = collections.deque(maxlen=num_frames)
        obs_shape = env.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0.0, high=1.0,
            shape=(num_frames, *obs_shape),
            dtype=np.float32
        )

    def reset(self, **kwargs):
        result = self.env.reset(**kwargs)
        obs = result[0] if isinstance(result, tuple) else result
        for _ in range(self.num_frames):
            self.frames.append(obs)
        return self._get_obs()

    def step(self, action):
        result = self.env.step(action)
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        elif len(result) == 4:
            obs, reward, done, info = result
            terminated = done
            truncated = False
        else:
            raise ValueError(f"step() returned unexpected number of values: {len(result)}")
        self.frames.append(obs)
        return self._get_obs(), reward, terminated, truncated, info

    def _get_obs(self):
        return np.array(self.frames, dtype=np.float32)

class GymCompatibilityV0(gym.Wrapper):
    def step(self, action):
        result = self.env.step(action)
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        elif len(result) == 4:
            obs, reward, done, info = result
        else:
            raise ValueError(f"step() returned unexpected number of values: {len(result)}")
        return obs, reward, done, info

    def reset(self, **kwargs):
        result = self.env.reset(**kwargs)
        if isinstance(result, tuple) and len(result) == 2:
            obs, _ = result
        else:
            obs = result
        return obs

def create_mario_env():
    env = gym.make('SuperMarioBros-v3', apply_api_compatibility=True, render_mode="rgb_array")
    env = JoypadSpace(env, SIMPLE_MOVEMENT)
    env = GrayScaleObservation(env, keep_dim=True)
    env = ResizeObservation(env, (84, 84))
    env = NormalizeObservation(env)
    env = FixedFrameStack(env, 4)
    env = GymCompatibilityV0(env)
    return env

### ✅ Paso 3: Conversión a entornos TF

In [ ]:


from tf_agents.environments import tf_py_environment, suite_gym

train_py_env = create_mario_env()
eval_py_env = create_mario_env()

train_env = tf_py_environment.TFPyEnvironment(suite_gym.wrap_env(train_py_env))
eval_env = tf_py_environment.TFPyEnvironment(suite_gym.wrap_env(eval_py_env))

print("✅ Entorno Mario Bros creado correctamente")
print("Especificación de observación:", train_env.observation_spec())
print("Especificación de acción:", train_env.action_spec())


### ✅ Paso 4: Definición de redes actor/critic

In [ ]:
from tf_agents.networks import actor_distribution_network, value_network
from tf_agents.trajectories import time_step as ts

observation_spec = train_env.observation_spec()
action_spec = train_env.action_spec()
time_step_spec = ts.time_step_spec(observation_spec)

conv_layer_params = [(32, (8, 8), 4), (64, (4, 4), 2), (64, (3, 3), 1)]
fc_layer_params = [512]

actor_net = actor_distribution_network.ActorDistributionNetwork(
    input_tensor_spec=observation_spec,
    output_tensor_spec=action_spec,
    conv_layer_params=conv_layer_params,
    fc_layer_params=fc_layer_params
)

value_net = value_network.ValueNetwork(
    input_tensor_spec=observation_spec,
    conv_layer_params=conv_layer_params,
    fc_layer_params=fc_layer_params
)

### ✅ Paso 5: Inicialización del agente PPO

In [ ]:
from tf_agents.agents.ppo import ppo_agent
import tensorflow as tf

strategy = tf.distribute.get_strategy()
with strategy.scope():
    optimizer = tf.keras.optimizers.Adam(learning_rate=2.5e-4)
    agent = ppo_agent.PPOAgent(
        time_step_spec=time_step_spec,
        action_spec=action_spec,
        optimizer=optimizer,
        actor_net=actor_net,
        value_net=value_net,
        importance_ratio_clipping=0.2,
        normalize_observations=True,
        normalize_rewards=True,
        use_gae=True,
        num_epochs=3,
        train_step_counter=tf.Variable(0)
    )
    agent.initialize()

### ✅ Paso 6: Métricas, Replay Buffer y Driver

In [ ]:
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.metrics import tf_metrics
from tf_agents.drivers.dynamic_step_driver import DynamicStepDriver

replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    data_spec=agent.collect_data_spec,
    batch_size=train_env.batch_size,
    max_length=10000
)

train_metrics = [
    tf_metrics.NumberOfEpisodes(),
    tf_metrics.EnvironmentSteps(),
    tf_metrics.AverageReturnMetric(buffer_size=10),
    tf_metrics.AverageEpisodeLengthMetric()
]

collect_driver = DynamicStepDriver(
    env=train_env,
    policy=agent.collect_policy,
    observers=[replay_buffer.add_batch] + train_metrics,
    num_steps=2048
)

### ✅ Paso 7: Ciclo de entrenamiento

In [ ]:
from tqdm import trange

num_iterations = 10000
rewards_history = []

print("\U0001F3CB️ Iniciando entrenamiento PPO...")
for iteration in trange(num_iterations, desc="Entrenando PPO"):
    collect_driver.run()
    experience = replay_buffer.gather_all()
    train_loss = agent.train(experience)
    replay_buffer.clear()
    avg_return = train_metrics[2].result().numpy()
    rewards_history.append((iteration, avg_return))

### ✅ Paso 8: Graficar recompensa

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_training_rewards(rewards_history, title="Recompensa Promedio por Iteración", window=10):
    if not rewards_history:
        print("⚠️ La lista rewards_history está vacía.")
        return

    iterations, rewards = zip(*rewards_history)
    smoothed_rewards = []
    for i in range(len(rewards)):
        start = max(0, i - window + 1)
        smoothed_rewards.append(np.mean(rewards[start:i+1]))

    plt.figure(figsize=(12, 6))
    plt.plot(iterations, rewards, label="Reward cruda", alpha=0.4)
    plt.plot(iterations, smoothed_rewards, label=f"Reward suavizada (media móvil {window})", linewidth=2)
    plt.xlabel("Iteración")
    plt.ylabel("Recompensa Promedio")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()

plot_training_rewards(rewards_history)


### ✅ Paso 9: Guardar política entrenada

In [ ]:
from tf_agents.policies import policy_saver
import os

policy_dir = "ppo_mario_policy"
os.makedirs(policy_dir, exist_ok=True)
tf_policy_saver = policy_saver.PolicySaver(agent.policy)
tf_policy_saver.save(policy_dir)

### ✅ Paso 10: Comparación en video con política aleatoria y entrenada


In [ ]:
from tf_agents.policies import py_tf_eager_policy
from tf_agents.trajectories.time_step import TimeStep
import imageio
import base64
from IPython.display import HTML

def create_render_env():
    return create_mario_env()

def generate_video_from_policy(policy, filename="ppo_mario_eval.mp4", num_episodes=1):
    render_env = create_render_env()
    frames = []
    for _ in range(num_episodes):
        obs = render_env.reset()
        frames.append(render_env.render())
        done = False
        while not done:
            obs_tensor = tf.convert_to_tensor([obs], dtype=tf.float32)
            time_step = TimeStep(
                step_type=tf.convert_to_tensor([0], dtype=tf.int32),
                reward=tf.convert_to_tensor([0.0], dtype=tf.float32),
                discount=tf.convert_to_tensor([1.0], dtype=tf.float32),
                observation=obs_tensor
            )
            action_step = policy.action(time_step)
            result = render_env.step(action_step.action.numpy()[0])
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            frames.append(render_env.render())
    imageio.mimsave(filename, frames, fps=30)
    mp4 = open(filename, 'rb').read()
    encoded = base64.b64encode(mp4).decode('ascii')
    return HTML(f'<video autoplay loop controls><source src="data:video/mp4;base64,{encoded}" type="video/mp4"></video>')

def generate_video_random_policy(filename="random_policy_eval.mp4", num_episodes=1, max_steps=1000):
    render_env = create_render_env()
    frames = []
    possible_actions = [1, 2, 3, 4, 5, 6]
    for _ in range(num_episodes):
        obs = render_env.reset()
        render_env.step(1)
        obs, *_ = render_env.step(1)
        frames.append(render_env.render())
        done = False
        step_count = 0
        while not done and step_count < max_steps:
            step_count += 1
            action = np.random.choice(possible_actions)
            result = render_env.step(action)
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                done = terminated or truncated
            else:
                obs, reward, done, info = result
            frames.append(render_env.render())
    imageio.mimsave(filename, frames, fps=30)
    with open(filename, 'rb') as f:
        video_data = f.read()
    encoded = base64.b64encode(video_data).decode('ascii')
    return HTML(f'<video autoplay loop controls><source src="data:video/mp4;base64,{encoded}" type="video/mp4"></video>')

# ▶️ Ejecutar
generate_video_from_policy(agent.policy)
generate_video_random_policy()